# NaviRaksha — SHAP Feature Explainability
Explains RF model predictions using SHAP values
**Run this after training RF model**

In [ ]:
import shap
import joblib
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Load model
MODEL_PATH = '../models/trained/rf_model.pkl'
SCALER_PATH = '../models/trained/rf_features.pkl'
TEST_DATA = '../data/processed/test_real.csv'

model = joblib.load(MODEL_PATH)
print('✅ Model loaded')

df = pd.read_csv(TEST_DATA)
print(f'✅ Test data: {df.shape}')

In [ ]:
# Select feature columns (same as training)
feature_cols = ['distance_km', 'hour', 'is_monsoon', 'ambulance_type', 'violations_zone']

# Use available columns
feature_cols = [c for c in feature_cols if c in df.columns]
print('Features:', feature_cols)

X_test = df[feature_cols].values[:200]  # Use 200 samples for SHAP speed

# SHAP explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print('✅ SHAP values computed')

In [ ]:
# ── PLOT 1: Feature Importance Summary ──
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols,
                  show=False, plot_type='bar')
plt.title('SHAP Feature Importance — NaviRaksha RF Model', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('../models/trained/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: shap_importance.png')

In [ ]:
# ── PLOT 2: Beeswarm (impact direction) ──
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
plt.title('SHAP Beeswarm — Feature Impact on ETA Prediction', fontsize=13)
plt.tight_layout()
plt.savefig('../models/trained/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: shap_beeswarm.png')

In [ ]:
# ── PLOT 3: Distance dependence (most important feature) ──
if 'distance_km' in feature_cols:
    plt.figure(figsize=(8, 5))
    dist_idx = feature_cols.index('distance_km')
    shap.dependence_plot('distance_km', shap_values, X_test,
                         feature_names=feature_cols, show=False)
    plt.title('SHAP Dependence: Distance vs. ETA Impact')
    plt.tight_layout()
    plt.savefig('../models/trained/shap_distance_dep.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: shap_distance_dep.png')

In [ ]:
# ── Numeric Summary Table ──
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Mean |SHAP|': mean_abs_shap,
    'Rank': pd.Series(mean_abs_shap).rank(ascending=False).astype(int)
}).sort_values('Rank')

print('\n📊 Feature Importance Summary:')
print(importance_df.to_string(index=False))
importance_df.to_csv('../models/trained/shap_importance_table.csv', index=False)
print('\n✅ Saved: shap_importance_table.csv')
print('\n✅ All SHAP analysis complete! Plots saved in models/trained/')